In [ ]:
from IPython.display import Image, display

# Population, Infrastructure, and Nighttime Lights: A Spatial Regression Analysis Across Countries

**Authors:** Bouchra Daddaoui · Namuunzaya Barsbold · Amanda González Mejía  
**Repository:** https://github.com/namibarsbold/STATS201-Course-project  
**GitHub Pages:** https://namibarsbold.github.io/STATS201-Course-project/

---
## Research Question and Motivation

This project asks how population density and infrastructure jointly predict satellite-observed nighttime light intensity, and whether the population–light relationship varies across regions and countries in ways consistent with spatial electrification inequality. Nighttime lights are widely used as a remotely sensed proxy for economic activity and development when administrative statistics are noisy or missing; related work emphasizes that lights also reflect infrastructure access and spatial inequality, capturing uneven electrification, industrial concentration, and development disparities (Henderson et al., 2012; Falchetta et al., 2023).

Electricity is a foundational input shaping labor markets, productivity, and household welfare. Causal evidence from rural electrification expansions links electricity access to higher labor market participation, particularly increased female employment in Africa (Dinkelman, 2011). If lights proxy electrification, then spatial variation in lights plausibly reflects differences in economic opportunity.

This variation is spatially structured. Subnational light patterns have been used to measure within-country inequality, revealing regional disparities that national averages obscure (Lipscomb, 2013). Related work argues that population–development elasticities vary across spatial contexts as institutions and infrastructure change how demographic concentration translates into observed outcomes (Lessmann & Seidel, 2017).

---
## Data and Feature Engineering

We construct a tile-level panel for three country case studies (Brazil, China, and Morocco), covering 2014–2023. The unit of observation is tile × year on a consistent spatial grid within each country. While the dataset spans 2014–2023, the final regression results in this report focus on the 2023 cross-section to enable clean cross-country comparison. This project is a supervised regression task with the objective to model spatial variation in satellite-observed nighttime light intensity as a function of population density and infrastructure characteristics.

- **Task:** Cross-sectional regression (tile-level, 2023)
- **Target variable:** $\log(1 + \text{Light}_{it})$
- **Core features:**
  1. $\log(1 + \text{Pop}_{it})$
  2. Region-type indicators and population × region interactions
  3. $\log(\text{DistanceToUrbanCore}_{it})$
  4. $\log(\text{LocalUrbanDensity}_{it})$
  5. Centrality
- **Key assumptions:**
  1. Nighttime radiance is a proxy for electrification- and development-related activity.
  2. The log–log specification yields interpretable elasticities.
  3. Observations are spatially correlated within countries; inference therefore relies on robust standard errors and descriptive interpretation rather than causal identification.

### Remote Sensing and Population Data

Nighttime lights come from annual composites of the VIIRS Day/Night Band produced for low-light detection by NASA and NOAA and distributed via the Earth Observation Group at Colorado School of Mines. These products are processed composites (not raw photographs): annual composites filter sunlit/moonlit/cloudy observations, screen ephemeral lights/background noise, rely on the "vcm" configuration to exclude stray-light-impacted data, and apply outlier filtering to reduce biomass burning and other anomalous radiance. Population data are annual gridded estimates from WorldPop, produced via spatial disaggregation (often Random Forest–based dasymetric mapping) rather than direct census counts at each pixel (a key measurement limitation).

### Spatial Aggregation and Variable Construction

Rasters (annual nightlights and annual population) are aggregated into fixed spatial tiles to form a panel dataset. The workflow describes adaptive tile sizing (256–1024 pixels; 60–80 tiles along the short dimension) to manage computation while keeping a stable within-country panel structure. Tiles are filtered for data quality by excluding observations below a valid-pixel threshold (e.g., <500 valid pixels). The resulting panel is reported as 30,442 tile-year observations across the three countries.

In [ ]:
# Log population vs log nightlights — core relationship motivating the study
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(mpimg.imread('../figures/week5_outputs/Brazil_2020_scatter_log_pop_vs_log_nl.png'))
axes[0].axis('off')
axes[0].set_title('Brazil (2020)', fontsize=11)
axes[1].imshow(mpimg.imread('../figures/week5_outputs/Morocco_2020_scatter_log_pop_vs_log_nl.png'))
axes[1].axis('off')
axes[1].set_title('Morocco (2020)', fontsize=11)
plt.suptitle('Figure 1. Log Population vs Log Nighttime Light — Raw Relationship', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

*Figure 1. Log population vs log nighttime light intensity at the tile level (2020). Both Brazil and Morocco show a clear positive association — tiles with higher resident populations tend to emit more light. However, substantial scatter around the trend line indicates that population density alone does not fully account for spatial variation in brightness. Outliers above the regression line likely reflect industrial zones, tourist areas, or resource-extraction sites with high radiance relative to permanent residents. Brazil shows a tighter overall relationship with a pronounced upper-right cluster of dense, bright metropolitan tiles; Morocco displays more heterogeneity, consistent with coastal concentration of economic activity and large dim interior regions. This scatter, combined with the range of light values at any given population level, motivates the regime-stratified interaction approach used throughout this study.*

### Region Typology

Annual lights and population rasters are aggregated into fixed tiles to form a panel. Tile sizing is adaptive (256–1024 pixels; 60–80 tiles along the short dimension) to manage computation while preserving within-country structure. Tiles are filtered for data quality (e.g., dropping tiles with fewer than 500 valid pixels). The resulting dataset contains 30,442 tile-year observations across the three countries. Each tile-year is classified into one of five spatial regimes based on the joint distribution of population density and light intensity within each country-year. Classification uses country-year–specific median thresholds for both variables, generating a bivariate high/low scheme:

- **urban_core:** high population, high light
- **dense_dim:** high population, low light
- **bright_sparse:** low population, high light
- **mixed:** intermediate combinations near thresholds
- **empty_or_rural:** low population, low light

Thresholds are computed separately for each country-year to preserve relative spatial structure. The typology is fully data-driven and not externally validated against administrative urban–rural definitions. Because it is constructed from the same variables used in the regression, it should be interpreted as descriptive stratification rather than as a causal treatment.

In [ ]:
# Region typology evolution over time — shows the 5-regime classification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(mpimg.imread('../figures/week5_outputs/Brazil_region_typology_over_time.png'))
axes[0].axis('off')
axes[0].set_title('Brazil', fontsize=11)
axes[1].imshow(mpimg.imread('../figures/week5_outputs/Morocco_region_type_shares_FIXED.png'))
axes[1].axis('off')
axes[1].set_title('Morocco', fontsize=11)
plt.suptitle('Figure 2. Region Typology Share Evolution (2014–2023)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

*Figure 2. Region typology share evolution across 2014–2023. In Brazil (left), spatial regime composition is relatively stable over the decade: the vast majority of tiles remain classified as "empty\_or\_rural," reflecting the country's large low-density interior, while "urban\_core" tiles constitute a consistently small but stable share concentrated in major metropolitan clusters. This regime stability suggests that Brazil's urban footprint expanded gradually rather than through distributed diffusion. In Morocco (right), regime shares shift more noticeably — a gradual reduction in "empty\_or\_rural" tiles and modest growth in "dense\_dim" and "mixed" tiles — consistent with ongoing peri-urban expansion and electrification around secondary cities over the decade. Temporal stability in Brazil supports using the typology as a structural covariate; the more dynamic pattern in Morocco motivates attention to temporal robustness.*

### Infrastructure Features

Infrastructure is proxied with spatial accessibility measures from tile geometry: (i) distance to urban core (isolation from the nearest urban-core tile), (ii) local urban density (count of urban-classified tiles within a fixed radius, capturing clustering/spillovers), and (iii) geographic centrality score (distance from the tile centroid to the national geographic centroid). These accessibility variables enter in logs for comparability with the log–log population/lights specification. We initially planned to include road density derived from OpenStreetMap via Geofabrik extracts (road length per tile area), but due to processing and harmonization constraints this variable is not included in the final specification and is treated as future work.

---
## Methods Modeling Strategy

We use interpretable regression (rather than high-capacity machine learning) because the goal is estimation and interpretation of regime-specific elasticities linking population density to nighttime light intensity. All models use a log–log structure, so slopes are interpretable as elasticities: a 1% increase in population density is associated with a $\beta$% change in lights, holding other regressors constant.

### Baseline Models — Pooled Log–Log Model

The simplest baseline estimates the overall association between population and light:

$$\log(1 + \text{Light}_{it}) = \alpha + \beta \log(1 + \text{Pop}_{it}) + \varepsilon_{it}$$

This specification establishes the aggregate population–light relationship without allowing spatial heterogeneity.

### Regime-Interaction Model

To allow elasticities to vary across spatial development regimes, we estimate:

$$\log(1 + \text{Light}_{it}) = \alpha + \beta \log(1 + \text{Pop}_{it}) + \gamma_r + \delta_r \bigl(\log(1 + \text{Pop}_{it}) \times R_{it}\bigr) + \varepsilon_{it}$$

where $r$ indexes region type. In this interaction design, the regime-specific elasticity for regime $r$ is given by $\beta + \delta_r$. This specification captures heterogeneity in how population translates into light across urban cores, dense-but-dim regions, bright-sparse areas, mixed tiles, and empty/rural tiles.

In [ ]:
# Regime-specific interaction slopes — directly illustrates the interaction model
display(Image('../figures/interaction_slopes_2023.png', width=720))

*Figure 3. Population–light interaction slopes by spatial regime (2023). Each bar shows the regime-specific elasticity — the percentage change in nighttime light intensity associated with a 1% increase in population density within that spatial type. Elasticities range from near-zero in "empty\_or\_rural" tiles (Brazil: 0.014; China: 0.017; Morocco: 0.015) to well above 0.4 in "urban\_core" tiles (Brazil: 0.471; China: 0.696; Morocco: 0.544). This sharp cross-regime variation confirms that the population–light relationship is strongly nonlinear in spatial context: demographic growth in low-access regions yields almost no increase in radiance, while equivalent growth in dense urban systems translates into large brightness gains. China's particularly high urban-core elasticity (0.696) suggests a strong "urban multiplier" effect — dense built form, shared networks, and industrial concentration appear to amplify population–light coupling far beyond what population size alone predicts. The near-zero rural elasticities across all three countries indicate that unelectrified or weakly electrified areas cannot convert population increases into observable light, highlighting the role of infrastructure access as the binding constraint in those regions.*

### Final Model

The final model extends the regime-interaction baseline by incorporating spatial accessibility variables that proxy infrastructure conditions. These variables are added additively while retaining regime-specific elasticities. The final specification is:

$$\log(1 + \text{Light}_{it}) = \alpha + (\beta + \delta_r)\log(1 + \text{Pop}_{it}) + \lambda_1 \log(\text{DistanceToUrbanCore}_{it}) + \lambda_2 \log(\text{LocalUrbanDensity}_{it}) + \lambda_3 \text{Centrality}_{it} + \gamma_r + \varepsilon_{it}$$

where $r$ denotes region type. Models are estimated by OLS. Key design choices that can affect estimates include tile sizing, valid-pixel thresholds, $\log(1+x)$ transforms, and median-based regime thresholds. Given heteroskedasticity in residuals (especially in high-light urban regions), we report heteroskedasticity-robust standard errors.

---
## Evaluation and Results

### Evaluation Framework

We evaluate models in-sample because the objective is structural interpretation of elasticities rather than predictive generalization. Neighboring tiles are spatially correlated, so random train/test splits would violate independence assumptions and overstate out-of-sample performance. Spatially blocked cross-validation is the appropriate method for formal out-of-sample evaluation, but it is not implemented in the current analysis.

### Metrics

We report three complementary performance measures. Firstly, R², which captures the share of variation in log-transformed nighttime light intensity explained by the model. Secondly, RMSE, which measures the typical magnitude of prediction error (in log units). Thirdly, AIC, an information criterion that compares model efficiency while penalizing additional covariates. Together, these metrics quantify explanatory power and relative model efficiency within the analyzed sample.

### Baseline Interaction Model Results

Fit is high across countries (R² = 0.879–0.941), and elasticities vary sharply by regime, indicating systematic spatial nonlinearity in population–light coupling. Empty/rural elasticities are near zero and statistically indistinguishable from zero (Brazil 0.014; China 0.017; Morocco 0.015). Substantively, marginal population increases in low-density areas yield negligible changes in radiance (a 10% population increase corresponds to roughly a 0.1–0.2% change in lights). Mixed tiles show small but statistically significant elasticities (Brazil 0.056; China 0.039; Morocco 0.083), consistent with population translating into brightness once baseline infrastructure is present (roughly 0.4–0.8% higher lights for a 10% population increase). Bright_sparse shows pronounced cross-country heterogeneity: Morocco has a large, highly significant elasticity (0.238), while Brazil (0.039) and China (0.030) are small and statistically weak—suggesting that "bright but low-density" areas capture different underlying structures across countries. Urban-core elasticities are large and precisely estimated (Brazil 0.471; China 0.696; Morocco 0.544). In China, a 10% increase in population in urban cores is associated with nearly a 7% increase in lights, consistent with an "urban premium" where dense built form and shared networks amplify the population–light relationship.

In [ ]:
# Regression summary table — quantitative results
display(Image('../figures/regression_summary_table_2023.png', width=750))

*Table 1. Model performance metrics (R², RMSE, AIC) for the baseline regime-interaction model and the infrastructure-augmented final model, by country, 2023 cross-section. R² is uniformly high across all countries and both specifications (0.879–0.954), indicating that population density and spatial regime account for the large majority of tile-level variation in nighttime light intensity. Adding infrastructure accessibility variables (distance to urban core, local urban density, centrality) yields modest but consistent R² gains — largest in Morocco (+0.013) and smallest in Brazil (+0.001) — and substantially reduces AIC in all three countries, indicating improved model efficiency relative to the baseline. The AIC drop is largest in China (ΔAIC = −28.1) and Morocco (−6.7), pointing to these countries as the contexts where infrastructure accessibility most distinctly shapes the light distribution beyond what regime type already captures. RMSE is lowest in Brazil and highest in China, consistent with China's wider absolute range of light values across its heterogeneous metropolitan and rural systems.*

In [ ]:
# Regime-specific elasticity comparison across countries
display(Image('../figures/final report/fig3_elasticity_comparison.png', width=720))

*Figure 4. Regime-specific population–light elasticities across Brazil, China, and Morocco (2023). Each cluster of bars represents a spatial regime, with country-specific estimates shown side by side. Urban-core elasticities are the largest across all three countries, confirming a consistent "urban premium" in population–light coupling. China's urban-core elasticity (0.696) is substantially higher than Brazil's (0.471) or Morocco's (0.544), suggesting that dense metropolitan systems in China convert population concentration into radiance with exceptional efficiency — possibly reflecting manufacturing and commercial density not present at the same scale elsewhere. At the other extreme, empty/rural elasticities approach zero in all three countries, indicating that population growth in low-density, low-light areas produces negligible additional brightness regardless of national context. Cross-country differences are most pronounced in "bright\_sparse" tiles: Morocco's elasticity (0.238) far exceeds Brazil's (0.039) and China's (0.030), suggesting that Morocco's sparse bright tiles — likely corresponding to irrigated agricultural zones, coastal tourism clusters, or mining sites — amplify population–light coupling through mechanisms absent from the other two countries. These cross-country contrasts highlight that what looks like a similar "light proxy" across countries may actually reflect structurally different underlying economic geographies.*

In [ ]:
# Predicted vs actual nighttime light — 2023 cross-section
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, country in zip(axes, ['Brazil', 'China', 'Morocco']):
    ax.imshow(mpimg.imread(f'../figures/week7_outputs/scatter_{country}_2023.png'))
    ax.axis('off')
    ax.set_title(country, fontsize=11)
plt.suptitle('Figure 5. Predicted vs Actual Nighttime Light (Final Model, 2023)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

*Figure 5. Predicted vs. actual log nighttime light intensity for the final infrastructure-augmented model (2023 cross-section). Points aligned with the 45-degree diagonal indicate accurate predictions; deviations reflect model error. Across all three countries, the bulk of observations cluster tightly around the line, consistent with the high in-sample R² values. A small number of extreme outliers at the upper end — primarily corresponding to ultra-bright tiles in industrial or metropolitan zones — show systematic underprediction, indicating that the constant-elasticity log–log specification does not fully capture nonlinear dynamics at the top of the light distribution. Brazil and Morocco show particularly tight central clustering; China exhibits somewhat greater spread at intermediate light levels, likely reflecting heterogeneity in mid-density tiles across China's vast geographic extent. The pattern of slight underprediction at the highest light values and marginal overprediction at very low values is consistent with a linear model in log space that imposes a uniform elasticity across the full distribution and cannot accommodate the steepening relationship observed in the brightest urban systems.*

### Infrastructure-Augmented Model Results

Adding distance to urban core, local urban density, and centrality modestly but consistently improves model performance relative to the baseline: ΔR² = +0.013 (Morocco), +0.005 (China), +0.001 (Brazil). AIC declines substantially, particularly in China (ΔAIC = −28.1) and Morocco (−6.7), indicating improved efficiency despite additional covariates. Conditioning on infrastructure shifts the role of population. In Morocco, log(pop) remains large and highly significant (0.242***), implying that a 10% population increase is associated with about a 2.4% increase in lights, holding infrastructure constant. In Brazil and China, the direct population coefficient becomes small and insignificant (0.003; 0.024), consistent with the idea that what looked like a "population effect" in simpler models is largely mediated by spatial accessibility and urban structure. Distance to urban centers is negative and significant in Brazil (−0.024***) and China (−0.027***): a 10% increase in distance predicts roughly 0.2–0.3% lower lights, conditional on population and regime. Local urban density is strongly negative in all countries (Morocco −0.749**; Brazil −0.786***; China −1.705***), capturing a steep gradient from dense urban systems to less developed peripheries. Centrality is significant and negative in China (−0.151***) and Morocco (−0.202**), but small and insignificant in Brazil. Because local urban density is mechanically related to the regime classification and broader spatial structure, the large negative coefficients should be interpreted as controlling for spatial stratification rather than as a standalone causal mechanism, as multicollinearity or suppression effects may partly influence magnitude.

In [ ]:
# R² comparison: baseline vs final model
display(Image('../figures/final report/fig1_r2_comparison.png', width=720))

*Figure 6. R² comparison between the baseline regime-interaction model and the infrastructure-augmented final model, by country (2023). The incremental gain in R² from adding distance to urban core, local urban density, and centrality is modest but consistent across all three countries. Morocco shows the largest gain (+0.013), consistent with the country's high geographic polarization — coastal concentration of cities and large interior distances to urban centers mean that spatial accessibility variables explain variance that population and typology alone cannot. China gains moderately (+0.005), suggesting that accessibility variables capture some additional spatial structure beyond regime classification. Brazil's gain is minimal (+0.001), indicating that its spatial structure is already well-described by regime type with population, and that infrastructure proxies add little incremental explanatory power. The consistently lower AIC across all three countries in the final model, despite added parameters, confirms that the infrastructure variables improve model efficiency — not just raw fit — and are worth including even where the R² change is small.*

---
## Diagnostics and Robustness Checks

### Diagnostic Assessment

Models capture large-scale spatial structure, but residual maps show remaining spatial clustering, suggesting omitted spatial processes. Residual-versus-fitted plots indicate modest dispersion reduction (especially in Morocco's infrastructure model), though heteroskedasticity persists in high-light urban regions (notably China), possibly due to nonlinear dynamics or unobserved metropolitan drivers. Robust standard errors address heteroskedasticity, but formal spatial autocorrelation tests (e.g., Moran's I) are needed. Key robustness sensitivities include the data-driven region typology (future check: alternative thresholds/base year regimes), aggregation sensitivity due to differing tile sizes (future check: alternative resolutions), and the influence of non-residential light sources (future check: excluding bright_sparse tiles or trimming extreme values). Out-of-sample evaluation requires spatially blocked validation.

In [ ]:
# Residual diagnostics
display(Image('../figures/final report/fig5_residuals.png', width=720))

*Figure 7. Residual-versus-fitted plots for Brazil (left, red), China (center, teal), and Morocco (right, gold) — final model, 2023 cross-section. Ideally, residuals should scatter randomly around zero with no systematic pattern. In practice, all three panels display a mild funnel shape: residual variance is larger at intermediate fitted values and tends to compress at the high end, indicating heteroskedasticity. This is most pronounced in China, where the highest-brightness metropolitan tiles carry large negative residuals (systematic underprediction), consistent with nonlinear dynamics in ultra-dense urban systems that a constant-elasticity log–log specification cannot fully capture. Morocco's residual spread is more symmetric, suggesting a better-calibrated fit across the light distribution. Brazil shows a small cluster of high-residual outliers likely corresponding to anomalous industrial tiles or resource-extraction zones. The persistence of heteroskedasticity across all three countries — despite the log transformation — motivates the use of heteroskedasticity-robust standard errors throughout and suggests that future work could benefit from quantile regression or spline extensions to relax the constant-elasticity assumption.*

In [ ]:
# Spatial residual maps — show where the model fails geographically
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, country in zip(axes, ['Brazil', 'China', 'Morocco']):
    ax.imshow(mpimg.imread(f'../figures/final report/figure_residual_maps_{country}_2020.png'))
    ax.axis('off')
    ax.set_title(country, fontsize=11)
plt.suptitle('Figure 8. Spatial Residual Maps (2020) — Remaining Clustering After Model Fit', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

*Figure 8. Spatial residual maps (2020) showing the geographic distribution of model prediction errors at the tile level. Warm colors indicate underprediction (actual light exceeds predicted); cool colors indicate overprediction (predicted exceeds actual). Systematic spatial clustering in residuals signals that not all spatial structure has been captured by population, regime type, and accessibility variables. In Brazil, residual clustering is visible in the northeastern interior (underprediction) and parts of the São Paulo metropolitan area, suggesting unmodeled regional heterogeneity — possibly reflecting labor market or policy differences between the developed Southeast and lagging Northeast. In China, pronounced residual clustering along the coastal manufacturing belt reflects agglomeration economies and industrial density that the population-and-accessibility framework does not capture; interior provinces show the opposite pattern of overprediction. In Morocco, residuals concentrate along the Casablanca–Rabat coastal corridor, consistent with economic agglomeration effects that exceed what the model's spatial variables can explain. These maps collectively highlight the need for additional controls — road density, industrial composition, policy-zone designations — in future model extensions, and they motivate formal spatial autocorrelation diagnostics (e.g., Moran's I) as a next step.*

In [ ]:
# Temporal R² stability — sensitivity check across 2014–2023
display(Image('../figures/final report/fig4_temporal_r2.png', width=720))

*Figure 9. Temporal stability of model R² across 2014–2023 (robustness check). The same final model specification is fitted separately to each annual cross-section, and resulting R² values are plotted over time. R² remains consistently high and stable across all three countries throughout the decade, with no systematic degradation, sharp discontinuities, or year-specific anomalies. This stability carries two important implications: first, it validates the use of the 2023 cross-section as a representative year — the results are not driven by idiosyncratic conditions in a single observation window; second, it indicates that the structural relationship between population density, spatial regime, infrastructure accessibility, and nighttime light is a persistent feature of spatial development rather than a transient artifact of a particular period. Minor year-to-year fluctuations are consistent with expected variation in nighttime radiance composites (caused by atmospheric conditions, sensor calibration, and seasonal filtering) and in WorldPop population estimates. The sustained high R² across a decade of satellite and population data provides strong support for the model's structural validity.*

---
## Interpretation and Substantive Takeaways

Population density is strongly associated with nighttime brightness across all three countries. However, the strength of this association varies systematically by spatial regime. In dense metropolitan systems, increases in population translate efficiently into greater luminosity, whereas in low-access regions, demographic growth produces little change in brightness. Cross-country comparisons therefore reflect differences in spatial structure rather than simply differences in national average brightness. For example, China's lower global coupling coefficient masks a sharp internal divide between highly efficient urban centers and weakly responsive rural regions. This highlights how national aggregates obscure regime-level inequality.

Adding infrastructure accessibility variables improves explanatory power most in geographically polarized systems, particularly Morocco and China. In Morocco, for instance, coastal concentration of lights and large interior distances to urban cores indicate that geographic isolation explains substantial additional variance beyond population. These findings suggest that electrification intensity is not purely demographic; it is mediated by spatial access and connectivity.

In [ ]:
# Infrastructure accessibility vs nighttime light
display(Image('../figures/final report/fig6_infra_vs_light.png', width=720))

*Figure 10. Infrastructure accessibility (distance to urban core) vs nighttime light intensity across tiles, by country. Each point represents a tile; the downward-sloping relationship indicates that tiles farther from the nearest urban center tend to emit less light, even after conditioning on population density and regime type. This negative gradient is steepest in Morocco and China, where geographic polarization is strong: beyond a certain distance from the urban network, tiles are systematically less bright regardless of resident population size. In Brazil, the gradient is present but flatter, consistent with a larger and more spatially distributed urban system where distance to a single urban core is a weaker predictor of connectivity and electrification. The persistence of this gradient after controlling for population and spatial regime supports interpreting geographic isolation as an independent dimension of electrification inequality — areas disconnected from urban infrastructure networks remain structurally underserved relative to what their population size would predict. This finding points to the importance of infrastructure investment in peripheral regions as a complement to, rather than a substitute for, urbanization-led development.*

In [ ]:
# Cross-country bivariate change — lights vs population change 2014–2023
display(Image('../figures/final report/figure_cross_country_bivar_change_2023.png', width=720))

*Figure 11. Cross-country bivariate change in population density and nighttime light intensity between 2014 and 2023. Each point represents a tile; axes show the change in log population (horizontal) and change in log nighttime light (vertical) over the decade. The upper-right quadrant (tiles gaining both population and brightness) captures urbanization-with-electrification; the lower-left quadrant (losing both) reflects depopulation or economic contraction in lagging regions. The slope of the point cloud captures how efficiently population growth converted into light gains across the decade. China's cloud is tightly clustered with a steep slope in areas that experienced large population gains, consistent with rapid urban electrification and infrastructure build-out driving bright outcomes from demographic growth. Brazil's cloud shows considerably more scatter, reflecting regional heterogeneity between a rapidly developing South and Southeast and a more stagnant Northeast interior. Morocco's cloud is concentrated near zero change for most tiles, with a small cluster of large gains in coastal and peri-urban corridors around Casablanca, Rabat, and Tangier. Together, these cross-decade dynamics illustrate that electrification inequality is not static — it reflects ongoing spatial restructuring driven by migration, infrastructure investment, and economic geography, all of which a longer panel could begin to disentangle causally.*

---
## Limitations and Future Work

### Limitations

Results are, overall, structural associations, not causal effects. The regime typology (constructed from population and lights) introduces potential endogeneity and should be treated as descriptive. Nighttime radiance is an imperfect proxy and can include non-residential sources (industrial sites, gas flares, road lighting), so lights should be interpreted as development/electrification-related activity rather than welfare. WorldPop estimates introduce measurement uncertainty, and residual clustering suggests omitted spatial processes (grid architecture, industrial composition, policy zones).

### Ethics and Data Governance

Inputs are aggregated satellite imagery and gridded population surfaces that do not identify individuals, but outputs can still be sensitive if interpreted as performance or deprivation measures; results should be communicated cautiously without normative labeling of regions. If OpenStreetMap-derived roads are incorporated later, attribution and ODbL compliance should be explicitly acknowledged.

### Future Work

Future research will incorporate exogenous road-density measures to better isolate infrastructure effects from typology-based spatial proxies. We also plan to address potential nonlinear dynamics in extreme urban cores through spline terms or quantile regression, as residual diagnostics suggest possible nonlinearity at high brightness levels. Future versions will also either harmonize tile resolution across countries or formally test elasticity stability under alternative spatial resolutions. Finally, extending the analysis across 2014–2023 will allow evaluation of whether population–light elasticities are stable over time or evolving alongside infrastructure expansion.

---
## References and AI Acknowledgment

Dinkelman, T. (2011). The effects of rural electrification on employment: New evidence from South Africa. *American Economic Review*, *101*(7), 3078–3108. https://doi.org/10.1257/aer.101.7.3078

Falchetta, G., Pachauri, S., Byers, E., Danylo, O., & Parkinson, S. C. (2020). Satellite observations reveal inequalities in the progress and effectiveness of recent electrification in Sub-Saharan Africa. *One Earth*, *2*(4), 364–379. https://doi.org/10.1016/j.oneear.2020.03.007

Henderson, J. V., Storeygard, A., & Weil, D. N. (2012). Measuring economic growth from outer space. *American Economic Review*, *102*(2), 994–1028. https://doi.org/10.1257/aer.102.2.994

Lessmann, C., & Seidel, A. (2017). Regional inequality, convergence, and its determinants – A view from outer space. *European Economic Review*, *92*, 110–132. https://doi.org/10.1016/j.euroecorev.2016.11.009

Lipscomb, M., Mobarak, A. M., & Barham, T. (2013). Development effects of electrification: Evidence from the topographic placement of hydropower plants in Brazil. *American Economic Journal: Applied Economics*, *5*(2), 200–231. https://doi.org/10.1257/app.5.2.200

---

**Datasets and tools:**  
VIIRS nighttime lights annual composites — Earth Observation Group, Colorado School of Mines ("Annual VNL V1"; vcm configuration)  
WorldPop population surfaces (RF/dasymetric redistribution; https://www.worldpop.org)  
Heteroskedasticity-robust inference: `statsmodels` `het_breuschpagan` and `get_robustcov_results`  
Regression metrics: `scikit-learn` R² and RMSE  

**GitHub repository:** namibarsbold/STATS201-Course-project

**AI Acknowledgment:** Regressions were brainstormed and edited with assistance from ChatGPT for correctness. Project infrastructure (repository structure, GitHub Pages setup, and reproducibility documentation) was designed with assistance from Claude (Anthropic).